# MAESTRO Post-Training Analysis

This notebook demonstrates downstream analyses you can perform after training a MAESTRO model on CyTOF data. It covers:

1. **Loading a trained model and data**
2. **Reconstruction visualization** — masking cells and comparing reconstructions
3. **Latent embedding extraction** — generating patient-level representations
4. **UMAP visualization** — embedding space colored by metadata (diagnosis, sex, age)
5. **Diagnosis classification** — logistic regression on latent embeddings
6. **Sex prediction** — binary classification from embeddings
7. **Age regression** — Ridge regression from embeddings
8. **Cell type proportion prediction** — predicting gated proportions from embeddings
9. **Batch effect analysis** — measuring robustness to marker intensity shifts
10. **Pool attention interpretation** — understanding which cell types the model attends to

**Note:** This notebook uses the toy/demo data included in this repository. Paths are relative to the repo root. Adjust paths and hyperparameters for your own data.

## 0. Setup & Imports

In [1]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import torch
import torch.nn.functional as F
import h5py
from collections import Counter
from tqdm.notebook import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import (
    confusion_matrix, accuracy_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from scipy.spatial.distance import pdist
from scipy import stats

# Add repo root to path so we can import MAESTRO modules
sys.path.insert(0, os.path.abspath('..'))
from data.cytof_dataset import CyTOFDataset
from models.MAESTRO import MAESTRO, MAESTROLightning, ruler_masking

warnings.filterwarnings('ignore')

# Plotting defaults
plt.style.use('default')
plt.rcParams.update({
    'figure.figsize': (5, 4),
    'font.family': 'sans-serif',
    'figure.dpi': 150,
    'axes.grid': False,
})

print('Imports complete.')

[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.
[KeOps] Warning : OpenMP library not found, it must be downloaded through Homebrew for apple Silicon chips
[KeOps] Warning : OpenMP support is not available. Disabling OpenMP.
Imports complete.


## 1. Load Data & Model

In [2]:
#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: mps


### 1a. Load the CyTOF dataset

The demo data contains two panels (dataA and dataB) with different markers. `CyTOFDataset` automatically discovers the shared markers across panels.

In [3]:
data_dir_A = os.path.abspath('../data/h5/dataA/')
data_dir_B = os.path.abspath('../data/h5/dataB/')

dataset = CyTOFDataset(data_dirs=[data_dir_A]) # CyTOFDataset(data_dirs=[data_dir_A, data_dir_B])

print(f'Number of samples: {len(dataset)}')
print(f'Shared markers ({len(dataset.shared_markers)}): {dataset.shared_markers}')

[Panel] dataA: 30 markers, 4 renamed
[Panel]   CCR7 -> CD197
[Panel]   CXCR5 -> CD185
[Panel]   CXCR3 -> CD183
[Panel]   CCR6 -> CD196
[Panel] Shared markers (30): ['CD11b', 'CD127', 'CD14', 'CD15', 'CD16', 'CD183', 'CD185', 'CD19', 'CD196', 'CD197', 'CD20', 'CD25', 'CD27', 'CD3', 'CD38', 'CD4', 'CD45', 'CD45RA', 'CD45RO', 'CD56', 'CD57', 'CD66b', 'CD69', 'CD8', 'FoxP3', 'HLA-DR', 'ICOS', 'IgD', 'PD-1', 'TIGIT']
Initialized dataset: 50 samples from 1 training directories
Shared markers (30): ['CD11b', 'CD127', 'CD14', 'CD15', 'CD16', 'CD183', 'CD185', 'CD19', 'CD196', 'CD197', 'CD20', 'CD25', 'CD27', 'CD3', 'CD38', 'CD4', 'CD45', 'CD45RA', 'CD45RO', 'CD56', 'CD57', 'CD66b', 'CD69', 'CD8', 'FoxP3', 'HLA-DR', 'ICOS', 'IgD', 'PD-1', 'TIGIT']
Number of samples: 50
Shared markers (30): ['CD11b', 'CD127', 'CD14', 'CD15', 'CD16', 'CD183', 'CD185', 'CD19', 'CD196', 'CD197', 'CD20', 'CD25', 'CD27', 'CD3', 'CD38', 'CD4', 'CD45', 'CD45RA', 'CD45RO', 'CD56', 'CD57', 'CD66b', 'CD69', 'CD8', 'FoxP3'

In [4]:
# Load all samples into memory
all_data = []
all_celltypes = []
all_names = []

for i in tqdm(range(len(dataset)), desc='Loading samples'):
    d, ct, name = dataset[i]
    all_data.append(d.unsqueeze(0))  # [1, n_cells, n_markers]
    all_celltypes.append(ct)
    all_names.append(name)

print(f'Loaded {len(all_data)} samples.')
print(f'Example sample shape: {all_data[0].shape}')
print(f'Discovered cell types: {dataset.get_all_cell_types()}')

Loading samples:   0%|          | 0/50 [00:00<?, ?it/s]

Loaded 50 samples.
Example sample shape: torch.Size([1, 21549, 30])
Discovered cell types: ['Basophil', 'Bcell', 'ClassicalMono', 'Eosinophil', 'MemoryCD4T', 'MemoryCD8T', 'NKcell', 'NaiveCD4T', 'NaiveCD8T', 'Neutrophil', 'NonClassicalMono', 'Treg', 'mDC', 'pDC']


### 1b. Load metadata

In [5]:
metadata = pd.read_csv('../data/csv/metadata.csv')

# Create a lookup from sample name to metadata
# The 'file' column contains paths like 'dataA/patient_001.csv'
# Sample names from h5 files are like 'patient_001'
metadata['sample_name'] = metadata['file'].apply(lambda x: os.path.splitext(os.path.basename(x))[0])
meta_dict = metadata.set_index('sample_name').to_dict('index')

print(f'Metadata columns: {list(metadata.columns)}')
print(f'Diagnoses: {metadata["Diagnosis"].unique()}')
metadata.head()

Metadata columns: ['file', 'panel', 'Diagnosis', 'Age', 'Sex', 'Treatment', 'sample_name']
Diagnoses: ['DiagnosisC' 'DiagnosisA' 'DiagnosisB']


,file,panel,Diagnosis,Age,Sex,Treatment,sample_name
0,dataA/patient_001.csv,A,DiagnosisC,31,M,TreatmentA,patient_001
1,dataA/patient_002.csv,A,DiagnosisA,74,F,TreatmentA,patient_002
2,dataA/patient_003.csv,A,DiagnosisC,68,M,TreatmentC,patient_003
3,dataA/patient_004.csv,A,DiagnosisB,20,M,TreatmentB,patient_004
4,dataA/patient_005.csv,A,DiagnosisC,48,M,TreatmentB,patient_005


### 1c. Load a trained MAESTRO model

Replace the paths below with your actual trained model checkpoint and config. If you haven't trained a model yet, see `train_maestro.sh` for an example training command.

In [ ]:
# ==========================================
# UPDATE THESE PATHS to your trained model
# ==========================================
CHECKPOINT_PATH = './output/training/ToyModel/model.ckpt'  # <-- UPDATE THIS
CONFIG_PATH = './output/training/ToyModel/config.pth'                # <-- UPDATE THIS

if not os.path.exists(CONFIG_PATH):
    print('WARNING: Model config not found. Please update CHECKPOINT_PATH and CONFIG_PATH above.')
    print('Skipping model loading. Downstream cells that require the model will not work.')
    model = None
else:
    config = torch.load(CONFIG_PATH, map_location=device)
    print('Model config:', {k: v for k, v in config.items()})

    # Load checkpoint — handles both Lightning and raw state_dict formats
    snapshot = torch.load(CHECKPOINT_PATH, map_location=device)
    if 'state_dict' in snapshot:
        state_dict = snapshot['state_dict']
        # Remove 'model.' prefix if present (from Lightning wrapper)
        state_dict = {k.replace('model.', '', 1): v for k, v in state_dict.items()}
    else:
        state_dict = snapshot

    model = MAESTRO(
        dim_input=config['dim_input'],
        dim_output=config['dim_output'],
        num_inds=config['num_inds'],
        dim_hidden=config['dim_hidden'],
        dim_latent=config['dim_latent'],
        num_heads=config['num_heads'],
        num_outputs=config['num_outputs'],
        ln=config['ln'],
        number_cells_subset=config['number_cells_subset'],
        student_temperature=config['student_temperature'],
        teacher_temperature=config['teacher_temperature'],
        sinkhorn_start=config.get('sinkhorn_start', 0),
    ).to(device)

    model.load_state_dict(state_dict, strict=False)
    model.eval()
    print(f'Model loaded successfully.')

Model config: {'dim_input': 13, 'dim_output': 13, 'num_inds': 16, 'dim_hidden': 384, 'dim_latent': 256, 'num_heads': 1, 'num_outputs': 40000, 'ln': True, 'number_cells_subset': 40000, 'initial_lr': 0.0001, 'min_lr': 1e-12, 'epochs': 1000, 'student_temperature': 0.11, 'teacher_temperature': 0.04, 'sinkhorn_start': 0, 'output_path': '/home/lematthe/MAESTRO/output/training/MAESTRO_BASE'}
Model loaded successfully.


---
## 2. Reconstruction Visualization

Mask a portion of cells and compare the model's reconstruction to the original data using UMAP.

In [ ]:
# Cell type colors for visualization
cell_type_colors = {
    'Neutrophil': '#0F4C81', 'MemoryCD4T': '#FF6F61', 'NaiveCD4T': '#009874',
    'MemoryCD8T': '#F5DF4D', 'NaiveCD8T': '#BF1932', 'Bcell': '#92A8D1',
    'ClassicalMono': '#88B04B', 'NonClassicalMono': '#9B1B30',
    'NKcell': '#E2583E', 'Treg': '#5A5B9F', 'mDC': '#F0C05A',
    'pDC': '#53B0AE', 'Eosinophil': '#F7CAC9', 'Basophil': '#5F4B8B',
}
default_color = '#AAAAAA'

# Pick a few samples to visualize
n_samples_viz = 3
sample_indices_viz = np.random.choice(len(all_data), n_samples_viz, replace=False)
mask_ratio = 0.5

fig, axes = plt.subplots(n_samples_viz, 4, figsize=(16, 4 * n_samples_viz))
col_titles = ['Cell Types (original)', 'Unmasked', 'Masked (target)', 'Predicted']

for row, idx in enumerate(sample_indices_viz):
    with torch.no_grad():
        input_data = all_data[idx].to(device)
        celltypes = all_celltypes[idx]
        name = all_names[idx]

        # Subsample for visualization speed
        n_cells = input_data.shape[1]
        viz_size = min(5000, n_cells)
        sub_idx = torch.randperm(n_cells)[:viz_size]
        input_sub = input_data[:, sub_idx, :]
        celltypes_sub = celltypes[sub_idx]

        # Generate mask using ruler masking
        mask = ruler_masking(input_sub, mask_ratio)
        B, N, D = input_sub.shape
        unmasked = input_sub[~mask].view(B, -1, D)
        masked_target = input_sub[mask].view(B, -1, D)

        # Forward pass through student encoder + decoder
        encoded, *_ = model.student.forward_encoder(unmasked)
        latent, _, _ = model.student.forward_pooling(encoded)
        pred_full = model.student.forward_decoder(latent, mask)

        # Sample predictions to match masked count
        n_masked = masked_target.shape[1]
        perm = torch.randperm(pred_full.shape[1])[:n_masked]
        pred = pred_full[:, perm, :]

        unmasked_np = unmasked.squeeze(0).float().cpu().numpy()
        target_np = masked_target.squeeze(0).float().cpu().numpy()
        pred_np = pred.squeeze(0).float().cpu().numpy()

    # Combined UMAP
    combined = np.vstack([unmasked_np, target_np, pred_np])
    reducer = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=42)
    emb = reducer.fit_transform(combined)

    n_u, n_t = len(unmasked_np), len(target_np)
    emb_u, emb_t, emb_p = emb[:n_u], emb[n_u:n_u+n_t], emb[n_u+n_t:]

    mask_cpu = mask.squeeze(0).cpu()
    unmasked_idx = (~mask_cpu).nonzero(as_tuple=True)[0]
    masked_idx = mask_cpu.nonzero(as_tuple=True)[0]
    ct_names_u = [dataset.get_cell_type_name(celltypes_sub[i].item()) for i in unmasked_idx]
    ct_names_m = [dataset.get_cell_type_name(celltypes_sub[i].item()) for i in masked_idx]
    all_ct = ct_names_u + ct_names_m

    xlim = [emb[:, 0].min() - 0.5, emb[:, 0].max() + 0.5]
    ylim = [emb[:, 1].min() - 0.5, emb[:, 1].max() + 0.5]

    # Plot cell types
    ax = axes[row, 0]
    colors = [cell_type_colors.get(ct, default_color) for ct in all_ct]
    ax.scatter(np.vstack([emb_u, emb_t])[:, 0], np.vstack([emb_u, emb_t])[:, 1],
               c=colors, s=1, alpha=0.5)
    ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_xticks([]); ax.set_yticks([])
    ax.set_ylabel(name, fontsize=9)
    if row == 0: ax.set_title(col_titles[0], fontsize=10)

    # Unmasked
    ax = axes[row, 1]
    ax.scatter(emb_u[:, 0], emb_u[:, 1], c='#FFB703', s=1, alpha=0.5)
    ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_xticks([]); ax.set_yticks([])
    if row == 0: ax.set_title(col_titles[1], fontsize=10)

    # Masked target
    ax = axes[row, 2]
    ax.scatter(emb_t[:, 0], emb_t[:, 1], c='#E63946', s=1, alpha=0.5)
    ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_xticks([]); ax.set_yticks([])
    if row == 0: ax.set_title(col_titles[2], fontsize=10)

    # Predicted
    ax = axes[row, 3]
    ax.scatter(emb_p[:, 0], emb_p[:, 1], c='#0077B6', s=1, alpha=0.5)
    ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_xticks([]); ax.set_yticks([])
    if row == 0: ax.set_title(col_titles[3], fontsize=10)

plt.suptitle(f'Reconstruction Visualization (mask ratio = {mask_ratio})', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. Latent Embedding Extraction

Generate a single latent vector per patient by passing all cells through the encoder and pooling layer.

In [ ]:
assert model is not None, 'Model not loaded.'

latent_dict = {}

for i in tqdm(range(len(all_data)), desc='Generating latent embeddings'):
    with torch.no_grad():
        input_data = all_data[i].to(device)
        x, *_ = model.student.forward_encoder(input_data)
        latent, _, _ = model.student.forward_pooling(x)
        latent_dict[all_names[i]] = latent.cpu()

# Build a DataFrame of latent features
latent_data = {}
for name, tensor in latent_dict.items():
    latent_data[name] = tensor.flatten().numpy()

latent_dim = list(latent_data.values())[0].shape[0]
latent_df = pd.DataFrame.from_dict(latent_data, orient='index')
latent_df.reset_index(inplace=True)
latent_df.columns = ['sample_name'] + [f'Feature_{i+1}' for i in range(latent_dim)]

print(f'Latent DataFrame shape: {latent_df.shape}')
latent_df.head()

---
## 4. UMAP Visualization of Latent Embeddings

Visualize the patient-level latent space, colored by diagnosis, sex, and age.

In [ ]:
# Merge latent features with metadata
merged_df = pd.merge(latent_df, metadata, on='sample_name', how='inner')
print(f'Merged DataFrame shape: {merged_df.shape}')

# Run UMAP on latent features
feature_cols = [c for c in merged_df.columns if c.startswith('Feature_')]
latent_matrix = merged_df[feature_cols].values

reducer = umap.UMAP(random_state=42, n_neighbors=15, min_dist=0.5)
embedding = reducer.fit_transform(latent_matrix)
merged_df['UMAP1'] = embedding[:, 0]
merged_df['UMAP2'] = embedding[:, 1]

In [ ]:
# Color by Diagnosis
diagnosis_colors = {
    'DiagnosisA': '#FF595E',
    'DiagnosisB': '#006DB8',
    'DiagnosisC': '#FFCA3A',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Diagnosis
ax = axes[0]
for diag, color in diagnosis_colors.items():
    mask = merged_df['Diagnosis'] == diag
    ax.scatter(merged_df.loc[mask, 'UMAP1'], merged_df.loc[mask, 'UMAP2'],
               c=color, label=diag, s=15, alpha=0.7)
ax.set_title('Colored by Diagnosis')
ax.legend(fontsize=8, markerscale=2)
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')

# Sex
sex_colors = {'M': '#88CCEE', 'F': '#CC6677'}
ax = axes[1]
for sex, color in sex_colors.items():
    mask = merged_df['Sex'] == sex
    ax.scatter(merged_df.loc[mask, 'UMAP1'], merged_df.loc[mask, 'UMAP2'],
               c=color, label=sex, s=15, alpha=0.7)
ax.set_title('Colored by Sex')
ax.legend(fontsize=8, markerscale=2)
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel('UMAP1')

# Age
ax = axes[2]
sc = ax.scatter(merged_df['UMAP1'], merged_df['UMAP2'],
                c=merged_df['Age'].astype(float), cmap='coolwarm', s=15, alpha=0.7)
plt.colorbar(sc, ax=ax, label='Age')
ax.set_title('Colored by Age')
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel('UMAP1')

plt.suptitle('MAESTRO Latent Embedding UMAP', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Diagnosis Classification

Train a logistic regression classifier on the latent embeddings to predict diagnosis, using stratified 5-fold cross-validation.

In [ ]:
# Prepare features and labels
feature_cols = [c for c in merged_df.columns if c.startswith('Feature_')]
X = merged_df[feature_cols].values
y = merged_df['Diagnosis'].values

classes = sorted(np.unique(y))
n_classes = len(classes)
print(f'Classes: {classes}, N={len(y)}')

# Stratified K-Fold cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_y_true, all_y_pred = [], []
fold_accuracies = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx])
    X_test = scaler.transform(X[test_idx])
    y_train, y_test = y[train_idx], y[test_idx]

    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    fold_accuracies.append(acc)
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)
    print(f'  Fold {fold+1}: Accuracy = {acc:.3f}')

print(f'\nMean Accuracy: {np.mean(fold_accuracies):.3f} +/- {np.std(fold_accuracies):.3f}')

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_y_true, all_y_pred, labels=classes)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=ax,
            vmin=0, vmax=1, square=True)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Diagnosis Classification (Normalized Confusion Matrix)')
plt.tight_layout()
plt.show()

---
## 6. Sex Prediction

Binary classification of sex from latent embeddings.

In [ ]:
sex_df = merged_df[merged_df['Sex'].isin(['M', 'F'])].copy()
X_sex = sex_df[feature_cols].values
y_sex = sex_df['Sex'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sex_true, sex_pred = [], []
sex_accs = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_sex, y_sex)):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_sex[train_idx])
    X_te = scaler.transform(X_sex[test_idx])

    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_tr, y_sex[train_idx])
    preds = clf.predict(X_te)

    acc = accuracy_score(y_sex[test_idx], preds)
    sex_accs.append(acc)
    sex_true.extend(y_sex[test_idx])
    sex_pred.extend(preds)

print(f'Sex Classification — Mean Accuracy: {np.mean(sex_accs):.3f} +/- {np.std(sex_accs):.3f}')

cm = confusion_matrix(sex_true, sex_pred, labels=['F', 'M'])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(4, 3))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=['F', 'M'], yticklabels=['F', 'M'], ax=ax,
            vmin=0, vmax=1, square=True)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Sex Prediction')
plt.tight_layout()
plt.show()

---
## 7. Age Regression

Ridge regression to predict age from latent embeddings.

In [ ]:
age_df = merged_df[pd.to_numeric(merged_df['Age'], errors='coerce').notna()].copy()
age_df['Age'] = age_df['Age'].astype(float)

X_age = age_df[feature_cols].values
y_age = age_df['Age'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)
age_true, age_pred = [], []

for train_idx, test_idx in kf.split(X_age):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_age[train_idx])
    X_te = scaler.transform(X_age[test_idx])

    reg = Ridge(alpha=1.0)
    reg.fit(X_tr, y_age[train_idx])
    preds = reg.predict(X_te)

    age_true.extend(y_age[test_idx])
    age_pred.extend(preds)

age_true = np.array(age_true)
age_pred = np.array(age_pred)
mae = mean_absolute_error(age_true, age_pred)
r2 = r2_score(age_true, age_pred)

print(f'Age Regression — MAE: {mae:.2f} years, R2: {r2:.3f}')

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(age_true, age_pred, alpha=0.6, s=20, edgecolors='k', linewidths=0.5)
lims = [min(age_true.min(), age_pred.min()) - 5, max(age_true.max(), age_pred.max()) + 5]
ax.plot(lims, lims, 'k--', alpha=0.5, label='Perfect prediction')
ax.set_xlabel('True Age')
ax.set_ylabel('Predicted Age')
ax.set_title(f'Age Regression (MAE={mae:.1f}, R²={r2:.2f})')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 8. Cell Type Proportion Prediction

Predict cell type proportions (computed from the gated labels) using the latent embeddings via Ridge regression.

In [ ]:
# Compute cell type proportions from the h5 gating labels
proportion_records = []
for i in range(len(all_data)):
    name = all_names[i]
    ct = all_celltypes[i].numpy()
    ct_names = dataset.get_cell_type_names(ct)
    counts = Counter(ct_names)
    total = sum(counts.values())
    props = {k: v / total for k, v in counts.items()}
    props['sample_name'] = name
    proportion_records.append(props)

prop_df = pd.DataFrame(proportion_records).fillna(0)
prop_df = prop_df.set_index('sample_name')

# Merge with latent features
prop_merged = pd.merge(latent_df, prop_df, left_on='sample_name', right_index=True, how='inner')

ct_cols = [c for c in prop_merged.columns if c not in feature_cols and c != 'sample_name']
print(f'Cell types to predict: {ct_cols}')

In [ ]:
# Ridge regression for each cell type proportion
X_prop = prop_merged[feature_cols].values
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_results = {}
for ct in ct_cols:
    y_ct = prop_merged[ct].values
    true_all, pred_all = [], []

    for train_idx, test_idx in kf.split(X_prop):
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_prop[train_idx])
        X_te = scaler.transform(X_prop[test_idx])

        reg = Ridge(alpha=1.0)
        reg.fit(X_tr, y_ct[train_idx])
        pred_all.extend(reg.predict(X_te))
        true_all.extend(y_ct[test_idx])

    mae_results[ct] = mean_absolute_error(true_all, pred_all)

# Plot MAE per cell type
sorted_ct = sorted(mae_results.keys(), key=lambda x: mae_results[x])
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(range(len(sorted_ct)), [mae_results[ct] for ct in sorted_ct], color='steelblue')
ax.set_yticks(range(len(sorted_ct)))
ax.set_yticklabels(sorted_ct, fontsize=8)
ax.set_xlabel('MAE (proportion)')
ax.set_title('Cell Type Proportion Prediction from Latent Embeddings')
plt.tight_layout()
plt.show()

---
## 9. Batch Effect Analysis

Measure embedding stability by applying random marker intensity shifts and comparing the resulting latent representations.

In [ ]:
assert model is not None, 'Model not loaded.'

def apply_marker_shift(input_data, gamma_shape=1.5, gamma_scale=3.0):
    """Apply random marker-specific intensity shifts."""
    X = input_data.float()
    B, N, D = X.shape
    shifts = torch.from_numpy(
        np.random.gamma(gamma_shape, gamma_scale, size=(1, 1, D))
    ).float().to(X.device)
    signs = torch.from_numpy(
        np.random.choice([-1, 1], size=(1, 1, D))
    ).float().to(X.device)
    return X + shifts * signs

# Generate shifted embeddings
shifted_latent_dict = {}
np.random.seed(42)

for i in tqdm(range(len(all_data)), desc='Computing shifted embeddings'):
    with torch.no_grad():
        input_data = all_data[i].to(device)
        shifted = apply_marker_shift(input_data)
        x, *_ = model.student.forward_encoder(shifted)
        latent, _, _ = model.student.forward_pooling(x)
        shifted_latent_dict[all_names[i]] = latent.cpu()

# Compute L2 distances between original and shifted
sample_names_sorted = sorted(latent_dict.keys())
distances = []
for name in sample_names_sorted:
    orig = latent_dict[name].flatten().numpy()
    shift = shifted_latent_dict[name].flatten().numpy()
    distances.append(np.linalg.norm(orig - shift))

print(f'Mean L2 distance (original vs shifted): {np.mean(distances):.4f}')
print(f'Std L2 distance: {np.std(distances):.4f}')

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(distances, bins=20, color='steelblue', edgecolor='white')
ax.axvline(np.mean(distances), color='red', linestyle='--', label=f'Mean = {np.mean(distances):.3f}')
ax.set_xlabel('L2 Distance')
ax.set_ylabel('Count')
ax.set_title('Embedding Stability Under Marker Shifts')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 10. Pool Attention Interpretation

Analyze which cell types receive the most attention during pooling, revealing what the model considers most informative.

In [ ]:
assert model is not None, 'Model not loaded.'

sample_names_attn = []
attention_results = []

for i in tqdm(range(len(all_data)), desc='Extracting pool attention'):
    with torch.no_grad():
        input_data = all_data[i].to(device)
        cell_types = all_celltypes[i]

        x, *_ = model.student.forward_encoder(input_data)
        _, _, pool_attn = model.student.forward_pooling(x)

        # pool_attn shape: [num_heads, 1, n_cells]
        # Average across heads and squeeze
        attn_weights = pool_attn.mean(dim=0).squeeze(0).cpu().numpy()  # [n_cells]

        ct_names = dataset.get_cell_type_names(cell_types)

        # Aggregate attention by cell type
        ct_attention = {}
        for ct, w in zip(ct_names, attn_weights):
            if ct not in ct_attention:
                ct_attention[ct] = []
            ct_attention[ct].append(w)

        # Mean attention per cell type
        ct_mean_attn = {ct: np.mean(ws) for ct, ws in ct_attention.items()}
        attention_results.append(ct_mean_attn)
        sample_names_attn.append(all_names[i])

# Build attention DataFrame
all_ct_set = sorted(set(ct for res in attention_results for ct in res))
attn_records = []
for i, name in enumerate(sample_names_attn):
    row = {'sample_name': name}
    for ct in all_ct_set:
        row[ct] = attention_results[i].get(ct, 0.0)
    attn_records.append(row)

attn_df = pd.DataFrame(attn_records)
print(f'Attention DataFrame shape: {attn_df.shape}')
attn_df.head()

In [ ]:
# Mean attention per cell type across all samples
mean_attn = attn_df[all_ct_set].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
colors = [cell_type_colors.get(ct, default_color) for ct in mean_attn.index]
ax.barh(range(len(mean_attn)), mean_attn.values, color=colors)
ax.set_yticks(range(len(mean_attn)))
ax.set_yticklabels(mean_attn.index, fontsize=8)
ax.set_xlabel('Mean Pool Attention Weight')
ax.set_title('Pool Attention by Cell Type')
plt.tight_layout()
plt.show()

In [ ]:
# Attention stratified by diagnosis
attn_meta = pd.merge(attn_df, metadata, on='sample_name', how='inner')

fig, axes = plt.subplots(1, len(diagnosis_colors), figsize=(5 * len(diagnosis_colors), 4), sharey=True)

for ax, (diag, color) in zip(axes, diagnosis_colors.items()):
    sub = attn_meta[attn_meta['Diagnosis'] == diag]
    mean_vals = sub[all_ct_set].mean().sort_values(ascending=True)
    ct_colors = [cell_type_colors.get(ct, default_color) for ct in mean_vals.index]
    ax.barh(range(len(mean_vals)), mean_vals.values, color=ct_colors)
    ax.set_yticks(range(len(mean_vals)))
    ax.set_yticklabels(mean_vals.index, fontsize=7)
    ax.set_title(f'{diag} (n={len(sub)})', fontsize=10)
    ax.set_xlabel('Mean Attention')

plt.suptitle('Pool Attention Stratified by Diagnosis', fontsize=13)
plt.tight_layout()
plt.show()

---
## Summary

This notebook demonstrated the key downstream analyses enabled by MAESTRO:

- **Reconstruction**: The model can reconstruct masked cells, preserving cell type structure.
- **Latent embeddings**: Each patient is compressed to a fixed-size vector capturing immune state.
- **Classification/Regression**: Latent embeddings support diagnosis, sex, and age prediction.
- **Cell type proportions**: The model implicitly learns cell type composition.
- **Batch robustness**: Embeddings are stable under simulated batch effects.
- **Attention**: Pool attention reveals which cell types are most informative.

For your own analyses, replace the toy data with your CyTOF datasets and update the model checkpoint paths.